In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import ipywidgets as widgets
from ipywidgets import interact

def density_and_derivs(x, components):
    # components: list of (mu, v, w) triples, weights normalized here
    wsum = sum(w for _, _, w in components)
    f = np.zeros_like(x)
    fp = np.zeros_like(x)
    for mu, v, w in components:
        w = w / wsum
        n = np.exp(-0.5 * (x - mu) ** 2 / v) / np.sqrt(2 * np.pi * v)
        f += w * n
        fp += w * (-(x - mu) / v) * n
    return f, fp

def find_maxima(components, xgrid):
    fp = density_and_derivs(xgrid, components)[1]
    roots = []
    sign = np.sign(fp)
    idx = np.where(np.diff(sign) != 0)[0]
    for i in idx:
        try:
            r = brentq(lambda x: density_and_derivs(x, components)[1], xgrid[i], xgrid[i + 1])
        except ValueError:
            continue
        eps = 1e-4
        if density_and_derivs(r - eps, components)[1] > 0 and density_and_derivs(r + eps, components)[1] < 0:
            roots.append(r)
    return roots


In [6]:
sigma_max = 2
sigmas = np.linspace(1e-3, sigma_max, 1000)
xgrid = np.linspace(-6, 6, 2000)
xgrid_img = np.linspace(-6, 6, 400)

def plot_fork(mu1=-1.0, mu2=1.0, sigma1=0.1, sigma2=0.1, w1=1.0, use_third=False, mu3=0.0, sigma3=0.1):
    base = [(mu1, sigma1, w1), (mu2, sigma2, 2.0 - w1)]
    if use_third:
        base.append((mu3, sigma3, 1.0))

    xs, ys = [], []
    for sigma in sigmas:
        components = [(mu, s ** 2 + sigma ** 2, w) for mu, s, w in base]
        for r in find_maxima(components, xgrid):
            xs.append(r)
            ys.append(sigma)

    dens = np.array([
        density_and_derivs(xgrid_img, [(mu, s ** 2 + sig ** 2, w) for mu, s, w in base])[0]
        for sig in sigmas
    ])

    plt.figure(figsize=(6, 8))
    plt.imshow(
        dens,
        extent=[xgrid_img[0], xgrid_img[-1], sigmas[0], sigmas[-1]],
        origin="lower",
        aspect="auto",
        cmap="viridis",
    )
    plt.colorbar(label="density")
    plt.scatter(xs, ys, s=2, c="red")
    plt.xlabel("x")
    plt.ylabel("noise sigma")
    title = f"mu1={mu1:.2f}, mu2={mu2:.2f}"
    if use_third:
        title += f", mu3={mu3:.2f}"
    plt.title(title)
    plt.ylim(0, sigma_max)
    plt.tight_layout()
    plt.show()

interact(
    plot_fork,
    mu1=widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.1, continuous_update=False),
    mu2=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, continuous_update=False),
    sigma1=widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, continuous_update=False),
    sigma2=widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, continuous_update=False),
    w1=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.05, description="w1 (w2=2-w1)", continuous_update=False),
    use_third=widgets.Checkbox(value=False, description="enable 3rd mode"),
    mu3=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, continuous_update=False),
    sigma3=widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, continuous_update=False),
)


interactive(children=(FloatSlider(value=-1.0, continuous_update=False, description='mu1', max=3.0, min=-3.0), …

<function __main__.plot_fork(mu1=-1.0, mu2=1.0, sigma1=0.1, sigma2=0.1, w1=1.0, use_third=False, mu3=0.0, sigma3=0.1)>

In [8]:
from scipy.spatial import cKDTree

def circle_density(theta, w_left):
    # density of arc-length measure on unit circle; left half = cos(theta) < 0
    left_mask = np.cos(theta) < 0
    return np.where(left_mask, w_left / np.pi, (1 - w_left) / np.pi)

n_theta = 800
theta_grid = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
dtheta = theta_grid[1] - theta_grid[0]
circle_xy = np.stack([np.cos(theta_grid), np.sin(theta_grid)], axis=1)  # (n_theta, 2)

grid_lim = 2.0
n_grid = 200
gx = np.linspace(-grid_lim, grid_lim, n_grid)
gy = np.linspace(-grid_lim, grid_lim, n_grid)
GX, GY = np.meshgrid(gx, gy)
G = np.stack([GX.ravel(), GY.ravel()], axis=-1)  # (M,2) query grid

def noised_density_2d(w_left, sigma):
    # p_sigma(x) = int f0(theta) N(x; circle_xy(theta), sigma^2 I) dtheta
    w = circle_density(theta_grid, w_left) * dtheta  # (n_theta,)
    dx = GX[..., None] - circle_xy[:, 0]  # (n_grid, n_grid, n_theta)
    dy = GY[..., None] - circle_xy[:, 1]
    sq = dx ** 2 + dy ** 2
    kernel = np.exp(-0.5 * sq / sigma ** 2) / (2 * np.pi * sigma ** 2)
    return kernel @ w  # (n_grid, n_grid)

def weighted_mixture_stats(X, Yc, weights, sigma, batch=4000):
    """X: (M,2) query points, Yc: (N,2) quadrature points, weights: (N,) summing to 1.
    Returns posterior mean mu (M,2), posterior covariance cov (M,2,2), log p_sigma(x) (M,)."""
    inv2s2 = 1.0 / (2.0 * sigma ** 2)
    log_norm_const = -np.log(2 * np.pi * sigma ** 2)
    log_w = np.log(weights)
    M = X.shape[0]
    mu = np.empty((M, 2))
    cov = np.empty((M, 2, 2))
    logp = np.empty(M)
    for start in range(0, M, batch):
        Xb = X[start:start + batch]
        diffs = Xb[:, None, :] - Yc[None, :, :]
        sqdist = np.sum(diffs ** 2, axis=-1)
        logk = -sqdist * inv2s2 + log_w[None, :]
        m = logk.max(axis=1, keepdims=True)
        w = np.exp(logk - m)
        wsum = w.sum(axis=1, keepdims=True)
        w /= wsum
        mu_b = w @ Yc
        c = Yc[None, :, :] - mu_b[:, None, :]
        cov_b = np.einsum('bn,bni,bnj->bij', w, c, c)
        mu[start:start + batch] = mu_b
        cov[start:start + batch] = cov_b
        logp[start:start + batch] = (m[:, 0] + np.log(wsum[:, 0])) + log_norm_const
    return mu, cov, logp

circle_tree = cKDTree(circle_xy)

def compute_ridge_points(w_left, sigma, epsilon):
    """Explicit epsilon-threshold ridge test: accept x if lambda2(x) < 0 and |g(x)^T v2(x)| < epsilon."""
    w = circle_density(theta_grid, w_left) * dtheta
    mu, cov, logp = weighted_mixture_stats(G, circle_xy, w, sigma)
    score = (mu - G) / sigma ** 2                  # g(x), Tweedie
    H = cov / sigma ** 4 - np.eye(2) / sigma ** 2   # Hessian

    eigvals, eigvecs = np.linalg.eigh(H)  # ascending: eigvals[:,0] <= eigvals[:,1]
    lambda2 = eigvals[:, 0]
    v2 = eigvecs[:, :, 0]                  # unit eigenvector, already normalized

    g_dot_v2 = np.sum(score * v2, axis=1)

    dist_to_curve, _ = circle_tree.query(G)

    is_ridge = (lambda2 < 0) & (np.abs(g_dot_v2) < epsilon) & (dist_to_curve <= 1.0 * sigma)
    return G[is_ridge]

pow2_levels = [2.0 ** k for k in range(-10, 1)]  # 2^-10 ... 2^0

def plot_circle_noise(w_left=0.5, sigma=0.2, epsilon_scale=0.01):
    sigma = max(sigma, 1e-3)
    epsilon = epsilon_scale / sigma  # |g^T v2| scales like 1/sigma, so threshold tracks it
    dens = noised_density_2d(w_left, sigma)
    ridge_pts = compute_ridge_points(w_left, sigma, epsilon)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(
        dens,
        extent=[-grid_lim, grid_lim, -grid_lim, grid_lim],
        origin="lower",
        cmap="viridis",
    )
    levels = [lv for lv in pow2_levels if dens.min() < lv < dens.max()]
    if levels:
        cs = ax.contour(GX, GY, dens, levels=levels, colors="white", linewidths=0.6)
        ax.clabel(cs, fmt=lambda v: f"2^{np.log2(v):.0f}", fontsize=6)
    ax.plot(circle_xy[:, 0], circle_xy[:, 1], "k-", lw=1, label="unit circle")
    if ridge_pts.shape[0]:
        ax.scatter(ridge_pts[:, 0], ridge_pts[:, 1], s=4, color="tab:red",
                   label=r"ridge pts ($\lambda_2<0$, $|g^\top v_2|<\varepsilon$)")
    ax.set_title(f"noised density in R^2, w_left={w_left:.2f}, sigma={sigma:.2f}, eps={epsilon:.3f}")
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

interact(
    plot_circle_noise,
    w_left=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.02, continuous_update=False),
    sigma=widgets.FloatSlider(value=0.2, min=0.02, max=1.5, step=0.02, continuous_update=False),
    epsilon_scale=widgets.FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001,
                                       description="eps_scale (eps=scale/sigma)", continuous_update=False),
)


interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='w_left', max=1.0, step=0.02…

<function __main__.plot_circle_noise(w_left=0.5, sigma=0.2, epsilon_scale=0.01)>